# PMAPS Workshop: IDAES-GTEP, Session A

Welcome! In this tutorial, we will demonstrate how to get started with IDAES-GTEP using the PJM 5-bus test case as an example.

Additional IDEAS-GTEP resources:
- https://github.com/IDAES/idaes-gtep
- https://idaes-gtep.readthedocs.io/en/latest/index.html

Other tools leveraged by IDEAS-GTEP:
- Prescient: https://github.com/grid-parity-exchange/prescient
- EGRET: https://github.com/grid-parity-exchange/egret

In [1]:
# Large amounts of warnings in this notebook are distracting; leaving this for now,
# unless we decide to resolve some of these warnings before the tutorial...

import logging
import warnings

logging.getLogger("pyomo").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

## Step 1: Reading in data

The `ExpansionPlanningData` class is our starting point. It allows us to define the temporal parameters of our model and reads in the appropriate data (EXPAND ON THIS, AND HOW IT CONTRASTS WITH DATA PROCESSING MODULE).

Its constructor takes the following arguments, all `int`:

| Name | Units | Default value | Description |
| --- | --- | --- | --- |
| `stages` | - | `2` | Number of investment periods |
| `num_reps` | - | `4` | Number of representative periods in each investment period |
| `len_reps` | Hours | `1` | Duration of each representative period |
| `num_commit` | - | `24` | Number of commitment periods in each representative period |
| `num_dispatch` | - | `1` | Number of dispatch periods in each commitment period |
| `duration_dispatch` | Minutes | `60` | Duration of each dispatch period |

Let's pick some values and create our `ExpansionPlanningData` object:

In [2]:
from gtep.gtep_data import ExpansionPlanningData

data_object = ExpansionPlanningData(
    stages=2,
    num_reps=2,
    len_reps=24,
    num_commit=2,
    num_dispatch=2,
)

Interactive Python mode detected; using default matplotlib backend for plotting.


Once the `ExpansionPlanningData` object is instantiated, we must point it to a directory containing data that will define our model setup. (DESCRIBE WHICH FILES ARE REQUIRED)

The IDAES-GTEP GitHub repository contains several example model setups, including one for the 5-bus test case. Below, we navigate to this directory and list the data files it contains:

In [3]:
from pathlib import Path

data_path = (Path() / ".." / "data" / "5bus").resolve()
for fpath in data_path.iterdir():
    print(fpath)

C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\branch.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\bus.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\DAY_AHEAD_load.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\DAY_AHEAD_renewables.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\gen.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\initial_status.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\README.md
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\REAL_TIME_load.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\REAL_TIME_renewables.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\reserves.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\simulation_objects.csv
C:\Users\agmoore\Documents\GitHub\idaes-gtep\gtep\data\5bus\timeseries_pointers.csv


[EXAMPLE] Try opening `gen.csv` above. You should see 8 different generators, each with an associated bus and unit type/technology, among other attributes.

Once we have the path to our data directory, we pass it into the `load_prescient` method of our `ExpansionPlanningData` object, which uses the data loader from production cost modeling platform Prescient.

The following table summarizes the arguments for this method, which determine how the Prescient data loader is used:

| Name | Type | Default value | Description |
| --- | --- | --- | --- |
| `data_path` | `pathlib.Path` or `str` | - | Path to directory containing the data |
| `representative_dates` | `list[?]` | `None` | ? |
| `representative_weights` | `dict[?]` | `{}` | ? |
| `options_dict` | `dict[?]` | `None` | Options passed to the Prescient data loader |

In [4]:
data_object.load_prescient(data_path)

Now that our data is loaded, it is stored under the `representative_data` attribute of the `ExpansionPlanningData` object. By printing its contents, we can see it consists of two EGRET `ModelData` objects, corresponding to the two representative periods that we defined in our `ExpansionPlanningData` constructor.

We can also explore the contents of these `ModelData` objects and see the data from our CSVs represented.

In [5]:
print(data_object.representative_data, "\n")

elements = data_object.representative_data[0].data["elements"]
print(elements.keys(), "\n")

for gen, gen_data in elements["generator"].items():
    print(f"{gen}    \t{gen_data['bus']}     \t{gen_data['generator_type']}    \t{gen_data['unit_type']}")

[<egret.data.model_data.ModelData object at 0x000002AF32550910>, <egret.data.model_data.ModelData object at 0x000002AF325016E0>] 

dict_keys(['bus', 'load', 'shunt', 'area', 'branch', 'generator', 'storage']) 

3_CT    	bus3     	thermal    	CT
10_STEAM    	bus10     	thermal    	STEAM
4_CC    	bus4     	thermal    	CC
4_STEAM    	bus4     	thermal    	STEAM
10_PV    	bus10     	renewable    	PV
2_RTPV    	bus2     	renewable    	RTPV
1_HYDRO    	bus1     	renewable    	HYDRO
4_WIND    	bus4     	renewable    	WIND


## 2. Creating the model

[Explain what inputs go into the model. We are setting up a simple case here but could, e.g., provide cost data as well]

In [ ]:
from gtep.gtep_model import ExpansionPlanningModel

mod_object = ExpansionPlanningModel(data=data_object)
# possible bug: if you pass config into the above call, the options aren't set. have
# to set them manually after creating the model

[Need some markdown here: explain what config options can be set here. 

For general code usability it would be good to have a documented list of all the config options; for the sake of this notebook, maybe just a few examples is enough]

In [ ]:
# print default config options (for now, let's just keep the default)
for a, b in mod_object.config.items():
    print(a, b)

include_investment True
include_commitment True
include_redispatch True
flow_model DC
time_period_subsets <abc.UninitializedConfigList object at 0x000002816C0AD040>
time_period_dict <pyomo.common.config.ConfigDict object at 0x000002816C07D090>
dispatch_randomization True
scale_loads True
scale_texas_loads False
thermal_generation False
renewable_generation False
storage False
transmission False
transmission_switching False
advanced_hydro False


Once we've configured everything, we call `create_model` to build the Pyomo model:

In [ ]:
mod_object.create_model()
print(type(mod_object.model))

Re-populating m.fuelCost, m.generatorInvestmentCost, m.fixedCost, and m.varCost for year 2025. These initialized parameters are overwritten using preprocessed data from m.mc.gen_data_target.
Cost data was not provided in m.mc instance (check DataProcessing for more details). Setting costs parameters to random values for now.


[    0.00] Creating GTEP Model
investmentStage[1].year = 2025


Re-populating m.fuelCost, m.generatorInvestmentCost, m.fixedCost, and m.varCost for year 2030. These initialized parameters are overwritten using preprocessed data from m.mc.gen_data_target.
Cost data was not provided in m.mc instance (check DataProcessing for more details). Setting costs parameters to random values for now.


investmentStage[2].year = 2030


Re-populating m.fuelCost, m.generatorInvestmentCost, m.fixedCost, and m.varCost for year 2035. These initialized parameters are overwritten using preprocessed data from m.mc.gen_data_target.
Cost data was not provided in m.mc instance (check DataProcessing for more details). Setting costs parameters to random values for now.


investmentStage[3].year = 2035
<class 'pyomo.core.base.PyomoModel.ConcreteModel'>


Note that the `ExpansionPlanningData` object is also stored on the model: 

In [ ]:
mod_object.model.data

In [ ]:
from pyomo.environ import SolverFactory, TransformationFactory

opt = SolverFactory("highs")

# TransformationFactory("gdp.bound_pretransformation").apply_to(mod_object.model)
# TransformationFactory("gdp.bigm").apply_to(mod_object.model)

opt.solve(mod_object.model)

{'Problem': [{'Lower bound': 13203418766.940912, 'Upper bound': 13203418766.940811, 'Number of objectives': 1, 'Number of constraints': nan, 'Number of variables': nan, 'Sense': 'minimize'}], 'Solver': [{'Status': 'ok', 'Termination condition': 'optimal', 'Termination message': 'TerminationCondition.convergenceCriteriaSatisfied'}], 'Solution': [OrderedDict({'number of solutions': 0, 'number of solutions displayed': 0})]}

### 3. Exploring results

In [ ]:
dispatch_blocks = (mod_object.model
    .investmentStage[1]
    .representativePeriod[1]
    .commitmentPeriod[1]
    .dispatchPeriod
)

for i in dispatch_blocks.index_set():
    b = dispatch_blocks[i]
    for key, val in b.thermalGeneration.items():
        print(key, val.value)

3_CT 20.0
10_STEAM 76.0
4_CC 100.0
4_STEAM 12.0
